# ECO462: Homework 7

## Question 1

In [316]:
# Import basic python libraries

import pandas as pd
import numpy as np
import math
import statsmodels.api as sm

df = pd.read_excel("../data/HW7 Data.xlsx")

df["Year: Month"] = df["Year: Month"].astype("Int64").astype(str)

# Convert to datetime index
df["date"] = pd.to_datetime(df["Year: Month"], format="%Y%m", errors="coerce")
df = df.set_index("date")

# Convert percent to decimals
df["Rm-Rf"] = df["Excess market return (%)"] / 100
df["rf"] = df["Riskfree rate (%)"] / 100
df["dy"] = df["Dividend yield (%)"] / 100
df["npy"] = df["Net payout yield (%)"] / 100

df = df.drop(df.index[-1])
df = df.sort_index()

In [317]:
# Question 1a Coding Portion

gamma = 5
sigma2 = 0.0019
mu = 0.0019

df_q1 = df.loc["1960-01-01":"2014-12-31"].copy()

x = np.clip(mu / (gamma * sigma2), 0, 1)

# Excess portfolio returns
df_q1["rp_excess"] = x * df_q1["Rm-Rf"]

# Sharpe ratio
SR_monthly = df_q1["rp_excess"].mean() / df_q1["rp_excess"].std()
SR_annual = np.sqrt(12) * SR_monthly

print("x =", x)
print("Annual Sharpe =", SR_annual)

x = 0.2
Annual Sharpe = 0.3945107091817782


In [318]:
# Question 1b Coding Portion

df = df.dropna(subset=["Rm-Rf"])
df = df.sort_index()

df["mu_hat"] = df["Rm-Rf"].rolling(60).mean()

df["x_t"] = (df["mu_hat"] / (gamma * sigma2)).clip(0, 1)

df["rp_excess"] = df["x_t"].shift(1) * df["Rm-Rf"]

df_q1b = df.loc["1960-01-01":"2014-12-31"].dropna()

SR_annual_1b = np.sqrt(12) * (
    df_q1b["rp_excess"].mean() /
    df_q1b["rp_excess"].std()
)

print("Annual Sharpe (1b) =", SR_annual_1b)

expected_values = df_q1b["mu_hat"].tolist()

print("\nList of Expected Values (mu_hat):")
print(expected_values)

Annual Sharpe (1b) = 0.2784956794737269

List of Expected Values (mu_hat):
[0.008906666666666665, 0.008598333333333331, 0.008353333333333332, 0.0075499999999999986, 0.007914999999999998, 0.007169999999999998, 0.006458333333333332, 0.006924999999999999, 0.005986666666666665, 0.0063149999999999986, 0.005924999999999998, 0.006461666666666666, 0.007999999999999998, 0.007966666666666665, 0.007341666666666666, 0.007343333333333332, 0.00861, 0.007516666666666665, 0.007181666666666665, 0.008139999999999998, 0.008638333333333333, 0.008979999999999997, 0.009661666666666664, 0.009104999999999999, 0.009056666666666664, 0.009701666666666664, 0.009233333333333333, 0.0074249999999999984, 0.005408333333333332, 0.004119999999999998, 0.005056666666666665, 0.006263333333333332, 0.006389999999999998, 0.007101666666666665, 0.00853, 0.009349999999999999, 0.009394999999999997, 0.009251666666666663, 0.009219999999999999, 0.009456666666666665, 0.009364999999999997, 0.00854333333333333, 0.007746666666666666, 0.

In [319]:
# Question 1c Coding Portion

df = df.sort_index()

mu_reg = np.full(len(df), np.nan)

start_index = 2

for t in range(start_index, len(df)):
    y_reg = df["Rm-Rf"].iloc[1:t].values
    X_reg = sm.add_constant(df["dy"].iloc[:t-1].values)

    valid = np.isfinite(y_reg) & np.all(np.isfinite(X_reg), axis=1)
    
    y_reg = y_reg[valid]
    X_reg = X_reg[valid]

    if len(y_reg) < 2:
        continue

    model = sm.OLS(y_reg, X_reg).fit()

    mu_reg[t] = model.params[0] + model.params[1] * df["dy"].iloc[t-1]

df["mu_reg"] = mu_reg

df["x_t"] = (df["mu_reg"] / (gamma * sigma2)).clip(0, 1)

df["rp_excess"] = df["x_t"].shift(1) * df["Rm-Rf"]

df_sample = df.loc["1960-01-01":"2014-12-31"].dropna(subset=["rp_excess"])

SR_annual = np.sqrt(12) * (
    df_sample["rp_excess"].mean()
    / df_sample["rp_excess"].std()
)

print("Annual Sharpe (1c) =", SR_annual)

expected_values_1c = df_sample["mu_reg"].tolist()

print("\nList of Expected Values for 1(c) (mu_reg):")
print(expected_values_1c)


Annual Sharpe (1c) = 0.3821828817318845

List of Expected Values for 1(c) (mu_reg):
[0.0031491951241625574, 0.0034202527193728646, 0.0034835446451193, 0.003592512643116136, 0.0036779273436987033, 0.0036442433231104786, 0.0035192487472200515, 0.003596971000255189, 0.0037095569584653775, 0.003998637115828472, 0.004020146026265098, 0.0038679721300620322, 0.0032743919890254897, 0.0030646738755001866, 0.0029755629469429765, 0.0028610528517653087, 0.0027848909322074683, 0.002853823087543177, 0.0028730190379311436, 0.0027719029970702386, 0.0029617616755082173, 0.0028203771951859245, 0.0028469531990610152, 0.003106464328932199, 0.0029093221199749057, 0.003041973497227699, 0.003171008690034908, 0.0029636521439500322, 0.003145895024618886, 0.003912718753233307, 0.004413828822547292, 0.003916411205924963, 0.0031927795478765165, 0.003586609771928059, 0.003703243855047415, 0.0033484257204606654, 0.0033749875969089384, 0.00307225566938419, 0.0033571656827400985, 0.0032488658430980446, 0.003099045644

In [320]:
# Question 1d Coding Portion

mu_reg = [np.nan] * len(df)

for t in range(start_index, len(df)):
    y = df["Rm-Rf"].iloc[:t]
    X = sm.add_constant(df["npy"].shift(1).iloc[:t])
    
    model = sm.OLS(y, X, missing="drop").fit()
    
    mu_reg[t] = model.params.iloc[0] + model.params.iloc[1] * df["npy"].iloc[t-1]

df["mu_reg"] = mu_reg

df["x_t"] = (df["mu_reg"] / (gamma * sigma2)).clip(0,1)

df["rp_excess"] = df["x_t"].shift(1) * df["Rm-Rf"]

df_sample = df.loc["1960-01-01":"2014-12-01"].dropna()

SR_annual = np.sqrt(12) * df_sample["rp_excess"].mean() / df_sample["rp_excess"].std()

print("Annual Sharpe (1d) =", SR_annual)

expected_values_1d = df_sample["mu_reg"].tolist()

print("\nList of Expected Values for 1(d) (mu_reg):")
print(expected_values_1d)

Annual Sharpe (1d) = 0.3575087367266877

List of Expected Values for 1(d) (mu_reg):
[0.0010881717947784078, 0.001233053446233633, 0.0012689071435394786, 0.0013221488089764331, 0.0014388052148444032, 0.0016899677996023484, 0.0016996438213374718, 0.0016707618041337437, 0.001933135651360933, 0.0020883897158218093, 0.0021324347245008334, 0.002194530268027077, 0.002492097807360115, 0.002371108005534758, 0.002402968955805041, 0.0011489067391823732, 0.0010484223525344047, 0.0011706127089634874, 0.0010914722443486858, 0.0010837790061586352, 0.001384281315717523, 0.001344254848269622, 0.0017915004343849843, 0.0021207120445275295, 0.003457512780616983, 0.0037209250802083104, 0.0039045384347755176, 0.00463538561125151, 0.005209301881273164, 0.0062606891034794835, 0.00843426831760629, 0.0037785660109443866, 0.00327825900666186, 0.0033997463349058335, 0.0032965337457125238, 0.003178477801607441, 0.005518538733920308, 0.005107332989985389, 0.0054711038007509245, 0.005445830258497113, 0.0051091029961

## Question 2

In [321]:
file_path = "../data/HW7 Data.xlsx"

xls = pd.ExcelFile(file_path)
df_raw = pd.read_excel(xls, "Problem 2")

df = df_raw.rename(columns={
    "Year: Month": "YearMonth",
    "Excess market return (%)": "RmRf_pct",
    "Riskfree rate (%)": "Rf_pct",
    "VIX (annual %)": "VIX_pct"
})

df["RmRf"] = df["RmRf_pct"] / 100.0
df["Rf"] = df["Rf_pct"] / 100.0
df["VIX"] = df["VIX_pct"] / 100.0

df = df.sort_values("YearMonth").reset_index(drop=True)

print("First few rows of cleaned data:")
print(df.head(), "\n")

start_ym = 199002
end_ym = 201412

mask_sample = (df["YearMonth"] >= start_ym) & (df["YearMonth"] <= end_ym)
df_sub = df.loc[mask_sample].copy().reset_index(drop=True)

print("Sample size (1990:2 to 2014:12):", len(df_sub))

gamma = 6.0
E_EXCESS = 0.0063

realized_mean = df_sub["RmRf"].mean()
print(f"Realized mean excess return in sample: {realized_mean:.6f}")
print(f"Fixed E[R_t - R_f,t] used in Q2:        {E_EXCESS:.6f}\n")

def compute_sharpe(returns):
    """
    returns: 1D array-like of monthly excess returns
    Returns (SR_month, SR_annual)
    """
    r = np.asarray(returns)
    mean_r = r.mean()
    std_r = r.std(ddof=1)
    SR_month = mean_r / std_r
    SR_annual = SR_month * math.sqrt(12.0)
    return mean_r, std_r, SR_month, SR_annual

sigma_uncond = df_sub["RmRf"].std(ddof=1)
var_uncond = sigma_uncond**2

print("=== Question 2(a): Constant sigma ===")
print(f"Unconditional sigma (RmRf): {sigma_uncond:.6f}")
print(f"Unconditional variance:     {var_uncond:.6f}")

x_const = E_EXCESS / (gamma * var_uncond)

x_const_clipped = max(0.0, min(1.0, x_const))

print(f"Unconstrained x*: {x_const:.6f}")
print(f"Constrained  x*: {x_const_clipped:.6f}\n")

Rp_e_a = x_const_clipped * df_sub["RmRf"]

mean_a, std_a, SR_month_a, SR_ann_a = compute_sharpe(Rp_e_a)

print("Results for 2(a):")
print(f"Mean monthly excess return: {mean_a:.6f}")
print(f"Std monthly excess return:  {std_a:.6f}")
print(f"Sharpe (monthly):          {SR_month_a:.6f}")
print(f"Sharpe (annualized):       {SR_ann_a:.6f}\n")

df["sigma_roll_12"] = df["RmRf"].rolling(window=12).std(ddof=1).shift(1)

df_sub["sigma_roll_12"] = df.loc[mask_sample, "sigma_roll_12"].values

print("=== Question 2(b): Rolling 12-month sigma ===")
print("Check first few sigma_roll_12 in subsample:")
print(df_sub[["YearMonth", "RmRf", "sigma_roll_12"]].head(), "\n")

sigma_roll = df_sub["sigma_roll_12"].to_numpy()
assert not np.isnan(sigma_roll).any(), "NaN found in rolling sigma for 2(b)."

x_roll = E_EXCESS / (gamma * sigma_roll**2)
x_roll_clipped = np.clip(x_roll, 0.0, 1.0)

print(f"x_roll min: {x_roll_clipped.min():.6f}, max: {x_roll_clipped.max():.6f}")
print(f"x_roll mean: {x_roll_clipped.mean():.6f}\n")

Rp_e_b = x_roll_clipped * df_sub["RmRf"].to_numpy()

mean_b, std_b, SR_month_b, SR_ann_b = compute_sharpe(Rp_e_b)

print("Results for 2(b):")
print(f"Mean monthly excess return: {mean_b:.6f}")
print(f"Std monthly excess return:  {std_b:.6f}")
print(f"Sharpe (monthly):          {SR_month_b:.6f}")
print(f"Sharpe (annualized):       {SR_ann_b:.6f}\n")

df["sigma_vix"] = (df["VIX"] / math.sqrt(12.0)).shift(1)

df_sub["sigma_vix"] = df.loc[mask_sample, "sigma_vix"].values

print("=== Question 2(c): VIX-based sigma ===")
print("Check first few sigma_vix in subsample:")
print(df_sub[["YearMonth", "VIX", "sigma_vix"]].head(), "\n")

sigma_vix = df_sub["sigma_vix"].to_numpy()
assert not np.isnan(sigma_vix).any(), "NaN found in VIX-based sigma for 2(c)."

x_vix = E_EXCESS / (gamma * sigma_vix**2)
x_vix_clipped = np.clip(x_vix, 0.0, 1.0)

print(f"x_vix min: {x_vix_clipped.min():.6f}, max: {x_vix_clipped.max():.6f}")
print(f"x_vix mean: {x_vix_clipped.mean():.6f}\n")

Rp_e_c = x_vix_clipped * df_sub["RmRf"].to_numpy()

mean_c, std_c, SR_month_c, SR_ann_c = compute_sharpe(Rp_e_c)

print("Results for 2(c):")
print(f"Mean monthly excess return: {mean_c:.6f}")
print(f"Std monthly excess return:  {std_c:.6f}")
print(f"Sharpe (monthly):          {SR_month_c:.6f}")
print(f"Sharpe (annualized):       {SR_ann_c:.6f}\n")

print("=== Summary of annualized Sharpe ratios for Q2 ===")
print(f"2(a) Constant sigma:        {SR_ann_a:.6f}")
print(f"2(b) Rolling 12m sigma:     {SR_ann_b:.6f}")
print(f"2(c) VIX-based sigma:       {SR_ann_c:.6f}")

First few rows of cleaned data:
   YearMonth  RmRf_pct  Rf_pct  VIX_pct    RmRf      Rf  VIX
0     198902     -2.25    0.61      NaN -0.0225  0.0061  NaN
1     198903      1.57    0.67      NaN  0.0157  0.0067  NaN
2     198904      4.33    0.67      NaN  0.0433  0.0067  NaN
3     198905      3.35    0.79      NaN  0.0335  0.0079  NaN
4     198906     -1.35    0.71      NaN -0.0135  0.0071  NaN 

Sample size (1990:2 to 2014:12): 299
Realized mean excess return in sample: 0.006536
Fixed E[R_t - R_f,t] used in Q2:        0.006300

=== Question 2(a): Constant sigma ===
Unconditional sigma (RmRf): 0.043342
Unconditional variance:     0.001878
Unconstrained x*: 0.558960
Constrained  x*: 0.558960

Results for 2(a):
Mean monthly excess return: 0.003653
Std monthly excess return:  0.024226
Sharpe (monthly):          0.150797
Sharpe (annualized):       0.522377

=== Question 2(b): Rolling 12-month sigma ===
Check first few sigma_roll_12 in subsample:
   YearMonth    RmRf  sigma_roll_12
0     19